# exp_6 · 04 target — only if H1 passed
Fills the recovered ball on accepted hidden graded frames (1 cm), runs exp_2's three full-data members,
decodes with exp_3's `L_ctrl` final config and provider hedge config, writes and validates both files, then
retrains the detector from scratch and rebuilds both files byte-identically. **Reads BAR-ATH: aggregates only;
never open this notebook executed.** Results: `results/target_candidates.json`.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp6_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_6')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp6_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

GUARD_FILE = 'guard_04.json'
guard = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / GUARD_FILE)
print('nothing written under exp_0-exp_5 / data since freezing:', guard['all_unchanged'],
      {k: v['unchanged'] for k, v in guard['folders'].items()})
if not guard['all_unchanged']:
    raise RuntimeError('Write guard changed: diagnose before continuing')

import exp6_decision as DE
import exp6_target as TG
log = make_log('04')
decision = C1.read_json(C.RESULTS / 'decision.json')
DE.require_h1(decision)

In [ ]:
first = TG.build(decision, C.RECOVERED, TG.CANDIDATES, log)
C.write_results(C.RESULTS / 'target_candidates.json', {'build': first, 'computed_at_utc': now()})
rebuild = TG.rebuild(decision, first, log)
C.write_results(C.RESULTS / 'target_candidates.json', {'build': first, 'rebuild': rebuild, 'computed_at_utc': now()})
decision.update({'files_written': True, 'files': {k: v['sha256'] for k, v in first['files'].items()},
                 'rebuild_byte_identical': rebuild['all_byte_identical']})
C.write_results(C.RESULTS / 'decision.json', decision)
print('fill:', first['fill'])
print('files:', first['files'])
print('rebuild byte-identical:', rebuild['all_byte_identical'])